First prototype and train in Python, then deploy in the browser or Node.js using TensorFlow.js.

1. Task Description Embedding
-- Used 'sentence-transformers' model to convert description into a numeric vector.

2. Skills Encoding
-- Convert skills into numeric form by
        Multi-hot encoding: each unique skill becomes a column (1 if task has it, 0 otherwise).

3. Category Encoding
-- Convert category into one-hot numeric vector.

In [1]:
import pandas as pd

df = pd.read_csv('../data/processed/tasks_with_id.csv')
df.head()


,task_id,description,category,skills
0,T00000,Implement user authentication,backend,spring boot
1,T00001,Optimize server performance,backend,asp.net
2,T00002,Manage database operations,backend,django
3,T00003,Implement user authentication,backend,api
4,T00004,Build a microservice,backend,kotlin


In [2]:
from sentence_transformers import SentenceTransformer
import pandas as pd

df = pd.read_csv('../data/processed/tasks_with_id.csv')
df.head()

# Load a small, fast pre-trained model that turns sentences into numbers
# Example: "Fix login bug" → [0.12, -0.34, ..., 0.08] (a 384-length vector)
# 'all-MiniLM-L6-v2' is just a fast model good for general English sentences
model = SentenceTransformer('all-MiniLM-L6-v2')

# Take all the task descriptions from the CSV
task_descriptions = df['description'].tolist()
# Example: ["Manage Agile workflows", "Improve website accessibility", "Monitor server health"]

# Convert each description into a vector of numbers
# These numbers capture the meaning of the task in a way the computer can understand
task_embeddings = model.encode(task_descriptions, show_progress_bar=True)

# Check one task embedding size: 384 numbers for one task
print("Embedding shape for one task:", task_embeddings[0].shape)

# Make sure we got an embedding for every task
print("Total embeddings:", len(task_embeddings))


Batches: 100%|██████████| 629/629 [00:13<00:00, 45.89it/s]

Embedding shape for one task: (384,)
Total embeddings: 20122


In [4]:
from sklearn.preprocessing import MultiLabelBinarizer

df = pd.read_csv('../data/processed/tasks_with_id.csv')
# Split skills if multiple per task (comma-separated)
df['skills_list'] = df['skills'].apply(lambda x: [s.strip() for s in x.split(',')])

mlb = MultiLabelBinarizer()
skills_encoded = mlb.fit_transform(df['skills_list'])

category_encoded = pd.get_dummies(df['category'])
print("Category one-hot shape:", category_encoded.shape)



Category one-hot shape: (20122, 13)
